# 06. Preparación de ventanas para modelos secuenciales

En este paso prepararemos la entrada de los modelos de la segunda fase, **sin entrenarlos todavía**. Cada ejemplo usa la historia de una máquina hasta una hora `t` y conserva las dos etiquetas a 24 horas del conjunto preparado.

Exploraremos **24, 72 y 168 horas**. Cada hora tendrá 13 canales: cuatro sensores, cinco indicadores horarios de error y cuatro tiempos desde reemplazos. Además, entregaremos el modelo de máquina y su antigüedad como entradas estáticas.

Conservamos las fechas y etiquetas del paso 04. No abrimos `prueba.csv` ni generamos resultados de prueba. Las tablas, ejemplos y verificaciones quedan en el notebook; no exportamos archivos adicionales.

## Antes de empezar: ¿por qué ahora usamos ventanas si antes calculamos promedios?

En la primera etapa, regresión logística y XGBoost recibían **una fila por máquina y hora**. Para darles información del pasado dentro de esa fila, creamos promedios, desviaciones y conteos de las horas anteriores. Esas columnas resumen la historia.

En esta segunda etapa, los modelos secuenciales pueden recibir **varias horas ordenadas como un solo ejemplo**. Así pueden aprender cómo cambian las señales, en lugar de depender únicamente de un resumen.

### Un ejemplo sencillo

Supongamos que la máquina 1 registró estos voltajes:

| Hora | Voltaje |
|---|---:|
| 08:00 | 160 |
| 09:00 | 170 |
| 10:00 | 180 |

**Primera etapa:** para predecir a las 10:00, una fila podía incluir el voltaje actual de 180 y el promedio de las últimas tres horas de 170, además de las otras variables. El modelo recibe esa fila con sus resúmenes.

**Segunda etapa:** una ventana de tres horas entregaría las tres mediciones juntas, en orden: **160 → 170 → 180**. Esas tres filas forman **un solo ejemplo**, con la misma etiqueta correspondiente a las 10:00. Aquí usamos tres horas solo para ilustrar; en el proyecto las ventanas serán de **24, 72 o 168 horas**.

El orden aporta información que un promedio puede ocultar:

```text
160 → 170 → 180: el voltaje sube. Promedio = 170.
180 → 170 → 160: el voltaje baja. Promedio = 170.
```

Ambas historias tienen el mismo promedio y la misma desviación estándar, pero evolucionan de forma diferente. Esto no significa que los modelos clásicos solo vieran promedios: también tenían el valor actual y otras variables. La diferencia es que ahora entregamos explícitamente la secuencia horaria completa.

### ¿De dónde sale la información y qué se conserva?

- **Los conjuntos preparados de entrenamiento y validación** nos dicen qué máquina y hora corresponde a cada ejemplo y cuáles son sus dos etiquetas.
- **Los archivos originales** permiten recuperar los sensores, errores y reemplazos de las horas que forman su ventana.
- **No se modifican los tres CSV preparados**, no se cambian las etiquetas ni la partición. Tampoco se abre prueba en este notebook.
- Las ventanas se preparan **en memoria** y se entregan por lotes; no hace falta guardar otro archivo gigante. Al cerrar la sesión se vuelven a construir ejecutando el código.

Los promedios de la primera fase no borraron las mediciones originales: fueron variables adicionales para la línea base. Ahora utilizamos la historia ordenada para estudiar otra forma de aprender.

**La comparación que buscamos es:** ¿los modelos secuenciales aprovechan esa historia mejor que los modelos clásicos con sus resúmenes? No está garantizado: si XGBoost sigue siendo mejor, también será un resultado válido. Esa comparación se hará más adelante, usando las mismas horas de validación.


## 1. Herramientas y lectura de los archivos

Los CSV preparados aportan únicamente las claves y las dos respuestas de entrenamiento y validación. La historia se reconstruye desde telemetría, errores, mantenimiento y catálogo.

Esto es necesario porque los conteos acumulados de error de la línea base no equivalen a indicadores horarios. Además, concatenar los conjuntos separados dejaría un hueco artificial al final de agosto, donde se retiraron horas por el horizonte de las etiquetas. Esas horas sí pueden aportar historia a septiembre.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from IPython.display import display

# Localizar el proyecto desde su raíz o desde Preprocesamiento.
actual = Path.cwd()
if (actual / "Datos").is_dir():
    ruta_repo = actual
elif (actual.parent / "Datos").is_dir():
    ruta_repo = actual.parent
else:
    raise FileNotFoundError("No se encontró la carpeta Datos.")

# Permitir que este notebook y los futuros modelos importen el módulo compartido.
ruta_codigo = str(ruta_repo / "Preprocesamiento")
if ruta_codigo not in sys.path:
    sys.path.insert(0, ruta_codigo)

from preparar_secuencias import (
    cargar_fuentes, construir_historia, normalizar_historia, preparar_estaticas,
    crear_series, crear_anclas, crear_datasets, preparar_secuencias,
    LONGITUDES, SENSORES, ERRORES, COMPONENTES, CLASES, SEMILLA,
    inicio, limite_entrenamiento, limite_validacion,
    canales, canales_errores, continuos, claves,
)

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
pd.set_option("display.max_columns", 14)
pd.set_option("display.max_rows", 25)

# Esta función conserva las fechas y las dos respuestas originales; no abre prueba.
datos, maquinas, objetivos = cargar_fuentes(ruta_repo)
print("Objetivos de entrenamiento:", len(objetivos["entrenamiento"]))
print("Objetivos de validación:", len(objetivos["validación"]))

Objetivos de entrenamiento: 580200
Objetivos de validación: 144000


Se utilizan las mismas **580,200 horas de entrenamiento y 144,000 de validación** de la primera fase. El catálogo contiene las entradas estáticas; las fallas no se consultan para crear canales predictivos. Sus respuestas ya vienen del paso 04.

## 2. Reconstruir los 13 canales por hora

| Canales | Significado y valores antes de normalizar |
|---|---|
| `volt`, `rotate`, `pressure`, `vibration` | Las cuatro mediciones originales de cada hora. |
| `error1_horario` a `error5_horario` | 1 si hubo al menos un registro de ese tipo en la hora; 0 si no. No son conteos de las últimas 24 horas. |
| `comp1_horas_desde_reemplazo` a `comp4_horas_desde_reemplazo` | Tiempo desde el último reemplazo del componente, estrictamente anterior a la hora consultada. Si no se conoce uno, queda vacío. |

No mezclamos máquinas. Un reemplazo de la misma hora no se utiliza hasta la siguiente, conservando la regla de la primera fase. Los errores de una hora se consideran observados en esa hora, suponiendo registros completos y disponibles. No rellenamos con eventos futuros.

In [2]:
# Reconstruir la historia con las mismas funciones que usarán los próximos modelos.
historia = construir_historia(datos, maquinas)
display(historia.loc[historia["datetime"] < limite_entrenamiento].head(5))
display(pd.DataFrame({"canal": canales, "tipo": ["sensor"] * 4 + ["indicador horario"] * 5 + ["tiempo desde reemplazo"] * 4}))

,machineID,datetime,volt,rotate,pressure,vibration,error1_horario,...,error3_horario,error4_horario,error5_horario,comp1_horas_desde_reemplazo,comp2_horas_desde_reemplazo,comp3_horas_desde_reemplazo,comp4_horas_desde_reemplazo
0,1,2015-01-01 06:00:00,176.217853,418.504078,113.077935,45.087686,0,...,0,0,0,456.0,5136.0,3696.0,4056.0
1,1,2015-01-01 07:00:00,162.879223,402.747490,95.460525,43.413973,0,...,0,0,0,457.0,5137.0,3697.0,4057.0
2,1,2015-01-01 08:00:00,170.989902,527.349825,75.237905,34.178847,0,...,0,0,0,458.0,5138.0,3698.0,4058.0
3,1,2015-01-01 09:00:00,162.462833,346.149335,109.248561,41.122144,0,...,0,0,0,459.0,5139.0,3699.0,4059.0
4,1,2015-01-01 10:00:00,157.610021,435.376873,111.886648,25.990511,0,...,0,0,0,460.0,5140.0,3700.0,4060.0


,canal,tipo
0,volt,sensor
1,rotate,sensor
2,pressure,sensor
3,vibration,sensor
4,error1_horario,indicador horario
5,error2_horario,indicador horario
6,error3_horario,indicador horario
7,error4_horario,indicador horario
8,error5_horario,indicador horario
9,comp1_horas_desde_reemplazo,tiempo desde reemplazo


La historia tiene una sola fila por máquina y hora y **13 canales**, sin las medias móviles de la línea base. Se conserva también el final de agosto como historia aunque esas horas no sean anclas etiquetadas de entrenamiento. Esto permite usar septiembre sin romper la continuidad.

## 3. Tratar faltantes y normalizar solo con entrenamiento

Usaremos **normalización global por canal**: a cada valor continuo se le resta la media y se divide entre la desviación estándar aprendidas en entrenamiento. Un cero normalizado significa estar en la media de entrenamiento; pueden aparecer valores negativos o mayores que 1. No es una escala limitada a 0–1.

Elegimos esta opción inicial porque mantiene un criterio común para las máquinas y los sensores están en la misma escala de origen entre modelos. No obliga a estimar estadísticas independientes por máquina; otras normalizaciones podrían compararse después usando validación.

Se normalizan los cuatro sensores y los cuatro tiempos desde reemplazos. Los cinco indicadores de error permanecen en 0/1. Si hubiera faltantes continuos, se usarían medianas de entrenamiento antes de normalizar. No se reemplaza la historia inexistente al inicio: ese caso se trata al formar las ventanas completas.

Para ajustar estadísticas usamos **solo las claves etiquetadas de entrenamiento**, no las de validación ni las últimas horas retiradas por el margen de agosto. Cada hora entra una vez, sin multiplicar su peso por aparecer en muchas ventanas.

In [3]:
# Ajustar solo con entrenamiento y transformar la historia sin reajustar en validación.
historia_normalizada, imputador, escalador, estadisticas = normalizar_historia(historia, objetivos)

# Mantener las tablas de auditoría para las comprobaciones explicadas al final.
historia_indice = historia.set_index(claves).sort_index()
indice_ajuste = pd.MultiIndex.from_frame(objetivos["entrenamiento"][claves])
base_ajuste = historia_indice.reindex(indice_ajuste)
ajuste_imputado = imputador.transform(base_ajuste[continuos])
display(estadisticas.round(3))
print("Horas usadas para ajustar:", len(base_ajuste))

Horas usadas para ajustar: 580200


,canal,faltantes_ajuste,mediana_entrenamiento,media_entrenamiento,desviacion_entrenamiento
0,volt,0,170.626,170.809,15.512
1,rotate,0,447.548,446.577,52.643
2,pressure,0,100.392,100.811,11.014
3,vibration,0,40.241,40.393,5.366
4,comp1_horas_desde_reemplazo,0,801.000,1397.481,1640.050
5,comp2_horas_desde_reemplazo,0,716.000,1361.062,1604.135
6,comp3_horas_desde_reemplazo,0,816.000,1409.075,1588.253
7,comp4_horas_desde_reemplazo,0,810.000,1408.464,1542.830


En estos archivos, los **ocho canales continuos no tienen faltantes** en las horas usadas para ajustar. A diferencia de las medias móviles de la línea base, aquí usamos directamente los sensores horarios y los tiempos desde reemplazos.

El imputador queda definido para conservar una regla reproducible si aparecieran vacíos, pero en esta ejecución no tuvo que sustituir ninguno. La normalización se ajustó con **580,200 horas de entrenamiento** y se aplicó sin recalcular estadísticas en validación. Los indicadores de error siguen siendo 0/1.

Por ejemplo, la media de voltaje de ajuste es aproximadamente **170.809** y su desviación **15.512**. Un voltaje igual a esa media se representa cerca de 0; uno una desviación por encima se representa cerca de 1. El tiempo desde reemplazos también se expresa en desviaciones estándar, no deja de representar historia pasada.

## 4. Preparar el modelo de máquina y la antigüedad

`model` se convierte en un código entero aprendido desde las máquinas presentes en entrenamiento. Ese código no indica gravedad ni orden: posteriormente será la entrada de una capa **embedding**, que aprenderá una representación numérica durante el entrenamiento del modelo. Todavía no se crea ni se entrena esa capa.

La antigüedad se normaliza con las máquinas de entrenamiento, contando cada máquina una sola vez. Se reserva el código 0 para un modelo de máquina desconocido; los conocidos comienzan en 1. Esta decisión permite manejar un modelo no visto sin cambiar el vocabulario al evaluar.

In [4]:
# Aprender categorías y escala de edad únicamente con las máquinas de entrenamiento.
catalogo, mapa_modelos, escalador_edad = preparar_estaticas(maquinas, objetivos)
ids_entrenamiento = objetivos["entrenamiento"]["machineID"].unique()
modelos_conocidos = list(mapa_modelos)
numero_codigos_modelo = len(mapa_modelos) + 1

display(pd.DataFrame({"modelo": modelos_conocidos, "codigo_embedding": [mapa_modelos[m] for m in modelos_conocidos]}))
display(catalogo.loc[ids_entrenamiento, ["model", "age", "modelo_codigo", "edad_normalizada"]].head(5))

,modelo,codigo_embedding
0,model1,1
1,model2,2
2,model3,3
3,model4,4


,model,age,modelo_codigo,edad_normalizada
machineID,,,,
1,model3,18,3,1.144551
2,model4,7,4,-0.743014
3,model3,8,3,-0.571417
4,model3,7,3,-0.743014
5,model3,2,3,-1.600998


El modelo queda representado por una categoría y la antigüedad por un número continuo. Las entradas estáticas se guardan separadas de las 13 señales horarias: no se repiten como canales en cada hora. Las dos etiquetas conservarán la misma interpretación que en la primera fase.

## 5. Localizar las ventanas completas de 24, 72 y 168 horas

Una ventana de longitud `L` termina en `t` y contiene exactamente las horas **t − (L − 1), …, t**. Por ejemplo, si `L = 24`, empieza 23 horas antes e incluye la hora actual: son 24 mediciones.

Cada ancla debe pertenecer al conjunto original y tener esa cantidad de historia de su propia máquina. Las horas anteriores a septiembre pueden formar parte de una ventana de validación; sus **etiquetas** no se incorporan a la entrada.

No construiremos todas las ventanas como un enorme arreglo duplicado: guardaremos la historia una sola vez y el índice final de cada ventana. Un arreglo completo de entrenamiento con 168 horas y 13 canales ocuparía varios GB. Después un `Dataset` entregará únicamente las ventanas solicitadas en cada lote.

In [5]:
# Guardar una sola historia por máquina e identificar sus ventanas completas.
series_maquina = crear_series(historia_normalizada, catalogo)
anclas, revision_ventanas = crear_anclas(objetivos, series_maquina)
display(revision_ventanas)
print("Las tres longitudes tienen las mismas horas de validación.")

Las tres longitudes tienen las mismas horas de validación.


,conjunto,L_horas,anclas_originales,ventanas_completas,excluidas_sin_historia,primera_ancla,ultima_ancla
0,entrenamiento,24,580200,577900,2300,2015-01-02 05:00:00,2015-08-30 23:00:00
1,validación,24,144000,144000,0,2015-09-01 00:00:00,2015-10-30 23:00:00
2,entrenamiento,72,580200,573100,7100,2015-01-04 05:00:00,2015-08-30 23:00:00
3,validación,72,144000,144000,0,2015-09-01 00:00:00,2015-10-30 23:00:00
4,entrenamiento,168,580200,563500,16700,2015-01-08 05:00:00,2015-08-30 23:00:00
5,validación,168,144000,144000,0,2015-09-01 00:00:00,2015-10-30 23:00:00


Quedan las siguientes ventanas completas de entrenamiento:

| Longitud | Ventanas conservadas | Horas excluidas por historia insuficiente |
|---|---:|---:|
| 24 horas | 577,900 | 2,300 |
| 72 horas | 573,100 | 7,100 |
| 168 horas | 563,500 | 16,700 |

Las tres opciones mantienen exactamente las **mismas 144,000 horas de validación**, incluyendo el 1 de septiembre a las 00:00, porque su historia puede comenzar en agosto.

Las exclusiones se producen al inicio de la telemetría de cada máquina. No se rellenan con ceros ni se inventa historia. La primera ancla de entrenamiento con 168 mediciones es el **8 de enero a las 05:00**. Al comparar longitudes habrá que recordar que cambia la cantidad de ejemplos de entrenamiento; si buscamos aislar ese efecto, podremos comparar también un subconjunto común de anclas, sin cambiar la partición temporal.

## 6. Entregar ventanas por lotes con PyTorch

Cada ejemplo entregará cinco elementos:

| Elemento | Forma de un lote de tamaño B | Tipo y significado |
|---|---|---|
| `dinamicas` | B × L × 13 | Números continuos y cinco indicadores de error. |
| `modelo` | B | Código entero para el futuro embedding. |
| `edad` | B × 1 | Antigüedad normalizada. |
| `binaria` | B × 1 | Respuesta 0/1, en formato decimal para la pérdida binaria. |
| `multiclase` | B | Código entero de clase: 0 ninguno, 1 comp1, …, 4 comp4. |

El `Dataset` es una forma de pedir un ejemplo por su posición. El `DataLoader` agrupa esos ejemplos. La permutación de ventanas de entrenamiento, si se activa más adelante, no cambia la partición temporal ni el orden de las horas dentro de cada ventana.

In [6]:
# Crear las vistas; los lotes siguientes solo comprueban formas, sin entrenar.
datasets = crear_datasets(anclas, series_maquina)
resumen_lotes = []
for longitud in LONGITUDES:
    
    for nombre in objetivos:
        dataset = datasets[longitud][nombre]

        # Solo obtener un lote pequeño para revisar sus formas, no entrenar un modelo.
        cargador = DataLoader(dataset, batch_size=4, shuffle=False, num_workers=0)
        lote = next(iter(cargador))
        assert tuple(lote["dinamicas"].shape) == (4, longitud, 13)
        assert torch.isfinite(lote["dinamicas"]).all()
        assert tuple(lote["edad"].shape) == (4, 1)
        assert tuple(lote["binaria"].shape) == (4, 1)
        assert tuple(lote["multiclase"].shape) == (4,)
        assert lote["modelo"].dtype == torch.int64
        resumen_lotes.append({"conjunto": nombre, "L": longitud, "forma_dinamicas": tuple(lote["dinamicas"].shape),
                             "forma_modelo": tuple(lote["modelo"].shape), "forma_edad": tuple(lote["edad"].shape)})

display(pd.DataFrame(resumen_lotes))

,conjunto,L,forma_dinamicas,forma_modelo,forma_edad
0,entrenamiento,24,"(4, 24, 13)","(4,)","(4, 1)"
1,validación,24,"(4, 24, 13)","(4,)","(4, 1)"
2,entrenamiento,72,"(4, 72, 13)","(4,)","(4, 1)"
3,validación,72,"(4, 72, 13)","(4,)","(4, 1)"
4,entrenamiento,168,"(4, 168, 13)","(4,)","(4, 1)"
5,validación,168,"(4, 168, 13)","(4,)","(4, 1)"


Las señales se entregan como **B × L × 13**. Las futuras LSTM y CNN-LSTM podrán usar ese orden; una CNN 1D normalmente requerirá intercambiar los dos últimos ejes a **B × 13 × L**. Ese cambio se hará dentro de la arquitectura sin cambiar el significado de los datos.

Solo se formaron lotes pequeños de comprobación. No se materializaron cientos de miles de ventanas ni se copiaron todos los datos para cada longitud.

### ¿Dónde queda la historia si no se modifica el CSV?

La historia que preparamos queda **temporalmente en la memoria de Python**, dentro de las variables del notebook. No se escribe encima de `entrenamiento.csv` ni de `validación.csv`.

| Variable del notebook | Qué contiene |
|---|---|
| `historia` | Una tabla con los sensores, indicadores de error y tiempos desde reemplazos de cada máquina y hora, antes de normalizar. |
| `historia_normalizada` | La misma historia, con los canales continuos normalizados usando parámetros de entrenamiento. |
| `series_maquina` | Esa historia organizada por máquina, junto con sus fechas y entradas estáticas, para buscar ventanas. |
| `datasets` | Objetos que recuperan una ventana cuando se solicita y entregan sus entradas y etiquetas. No guardan todas las ventanas duplicadas. |

**Ejemplo:** después de ejecutar las celdas anteriores, esta instrucción pide el primer ejemplo de entrenamiento con una ventana de 24 horas:

```python
ejemplo = datasets[24]["entrenamiento"][0]
```

En `ejemplo["dinamicas"]` hay **24 horas ordenadas × 13 canales**. El mismo ejemplo también contiene el código del modelo de máquina, la antigüedad y las dos etiquetas. El `DataLoader` agrupa varios ejemplos así para entregárselos al modelo.

Por tanto, las dos representaciones existen por separado:

```text
En el disco de tu computadora:
entrenamiento.csv → conserva las variables de la primera etapa.

En la memoria de Python:
historia y datasets → permiten obtener las ventanas de la segunda etapa.
```

Las mediciones horarias se leen de los archivos originales; **no se reconstruyen a partir de los promedios**. De los conjuntos preparados conservamos las claves de cada ejemplo y sus etiquetas.

El modelo puede recibir datos desde estas variables de Python: no necesita leer directamente un CSV. Si cierras o reinicias la sesión de Python, estas variables desaparecen de memoria. Para recuperarlas se ejecuta de nuevo el notebook. Los CSV permanecen intactos y no hace falta guardar otro archivo gigante.


## 7. Contrastar las ventanas y la historia con las fuentes

Revisaremos por cada longitud ejemplos iniciales y finales de entrenamiento y validación: fechas exactas, continuidad, ausencia de horas posteriores a `t` y etiquetas idénticas a las originales. También reconstruiremos un ejemplo de septiembre desde las fuentes para comprobar sensores, errores horarios y reemplazos.

Estas comprobaciones son sobre los datos, no sobre predicciones. Validación no vuelve a ajustar ningún parámetro.

In [7]:
verificaciones = []
for longitud in LONGITUDES:
    for nombre in objetivos:
        tabla = anclas[longitud][nombre]
        ejemplos = pd.concat([tabla.head(2), tabla.tail(2)]).drop_duplicates(claves)
        originales = objetivos[nombre].set_index(claves)

        for fila in ejemplos.itertuples(index=False):
            serie = series_maquina[fila.machineID]
            final = fila.posicion_final
            fechas = serie["horas"][final - longitud + 1:final + 1]
            assert len(fechas) == longitud
            assert fechas[-1] == fila.datetime
            assert fechas[0] == fila.datetime - pd.Timedelta(hours=longitud - 1)
            assert (fechas <= fila.datetime).all()
            assert ((fechas[1:] - fechas[:-1]) == pd.Timedelta(hours=1)).all()
            original = originales.loc[(fila.machineID, fila.datetime)]
            assert fila.etiqueta_binaria == original["etiqueta_binaria"]
            assert fila.etiqueta_multiclase == original["etiqueta_multiclase"]

        verificaciones.append({"conjunto": nombre, "L": longitud, "resultado": "Correcto"})

# La primera ventana larga de validación debe comenzar en agosto, no en septiembre.
elegida = anclas[168]["validación"].iloc[0]
serie = series_maquina[elegida["machineID"]]
final = int(elegida["posicion_final"])
fechas = serie["horas"][final - 167:final + 1]
assert fechas[0] < limite_entrenamiento
assert fechas[-1] == limite_entrenamiento
print("Ejemplo de ventana de validación:", fechas[0], "hasta", fechas[-1])

# Recuperar canales sin normalizar y contrastar con las cuatro fuentes de entradas.
for hora in [fechas[0], fechas[-1]]:
    fila = historia_indice.loc[(elegida["machineID"], hora)]

    # Contrastar también los sensores originales de esa máquina y hora.
    medicion = datos["telemetria"].loc[(datos["telemetria"]["machineID"] == elegida["machineID"])
                                      & (datos["telemetria"]["datetime"] == hora), SENSORES].iloc[0]
    assert np.allclose(fila[SENSORES].to_numpy(dtype=float), medicion.to_numpy(dtype=float))

    errores = datos["errores"].loc[(datos["errores"]["machineID"] == elegida["machineID"])
                                 & (datos["errores"]["datetime"] == hora)]
    for error in ERRORES:
        assert fila[error + "_horario"] == int((errores["errorID"] == error).any())

    for componente in COMPONENTES:
        previos = datos["mantenimiento"].loc[(datos["mantenimiento"]["machineID"] == elegida["machineID"])
            & (datos["mantenimiento"]["comp"] == componente) & (datos["mantenimiento"]["datetime"] < hora), "datetime"]
        esperado = (hora - previos.max()).total_seconds() / 3600 if not previos.empty else np.nan
        assert np.isclose(fila[componente + "_horas_desde_reemplazo"], esperado, equal_nan=True)

# El escalador recibió solo las horas de ajuste de entrenamiento, no la historia completa.
assert int(escalador.n_samples_seen_) == len(objetivos["entrenamiento"])
assert np.allclose(escalador.mean_, ajuste_imputado.mean(axis=0))
assert np.allclose(escalador.var_, ajuste_imputado.var(axis=0))
assert imputador.statistics_.shape[0] == 8

display(pd.DataFrame(verificaciones))

Ejemplo de ventana de validación: 2015-08-25 01:00:00 hasta 2015-09-01 00:00:00


,conjunto,L,resultado
0,entrenamiento,24,Correcto
1,validación,24,Correcto
2,entrenamiento,72,Correcto
3,validación,72,Correcto
4,entrenamiento,168,Correcto
5,validación,168,Correcto


Las verificaciones de fechas, continuidad y etiquetas pasaron para las seis combinaciones de conjunto y longitud. Las estadísticas coinciden con las calculadas en entrenamiento, y los ejemplos de errores y reemplazos coinciden con los registros originales.

La primera ventana de validación de **168 horas va del 25 de agosto a las 01:00 al 1 de septiembre a las 00:00**. Esto comprueba que se recuperó la historia de agosto sin incorporar horas posteriores al ancla. Las etiquetas siguen siendo las de septiembre; no se añadieron como canales de entrada.

Los lotes de ejemplo tienen formas **(4, 24, 13), (4, 72, 13) y (4, 168, 13)**. También se comprobaron las entradas estáticas y las dos salidas. No se ajustó ninguna arquitectura y no se abrió el conjunto de prueba.

## 8. Reproducir la preparación con el módulo compartido

**`preparar_secuencias.py` contiene la preparación de esta segunda etapa.** Es el equivalente al script del paso 04, pero prepara ventanas en memoria en lugar de guardar otros CSV.

Las funciones de las celdas anteriores se importan de ese archivo. El notebook explica y verifica cada paso; el `.py` permite usar exactamente las mismas operaciones desde los próximos notebooks. No hay que copiar este código ni mantener el notebook 06 abierto.

El módulo hace todo lo necesario para las entradas secuenciales:

1. Lee las claves y etiquetas de entrenamiento y validación, y recupera la historia de los originales.
2. Construye los 13 canales horarios, respetando cada máquina y el tiempo disponible.
3. Aprende la imputación, normalización y categorías únicamente con entrenamiento.
4. Localiza ventanas completas de 24, 72 y 168 horas sin cambiar etiquetas ni fechas de partición.
5. Devuelve los `Dataset`, los índices de las ventanas y los parámetros de preparación en memoria.

### Cómo se usará desde un notebook de modelos

Una vez localizada la raíz del proyecto e incluida `Preprocesamiento` en las rutas de importación, como en la primera celda de este notebook:

```python
from preparar_secuencias import preparar_secuencias

preparacion = preparar_secuencias(ruta_repo)
datos_entrenamiento = preparacion["datasets"][24]["entrenamiento"]
datos_validacion = preparacion["datasets"][24]["validación"]
```

`preparacion` es un diccionario: sus nombres permiten acceder a las ventanas y a los parámetros aprendidos. Los modelos usarán esos datos por lotes con `DataLoader`. Si cierras Python, se vuelven a crear con esa llamada.

### Cómo comprobar el módulo sin abrir el notebook

Desde la raíz del repositorio:

```bash
python Preprocesamiento/preparar_secuencias.py
```

El comando ejecuta la preparación y muestra el resumen de ventanas. **No guarda archivos ni deja variables disponibles para otra sesión**: para entrenar en otro notebook se importa y llama la función como en el ejemplo anterior.

Los conjuntos preparados de la primera fase deben existir previamente; si faltan, se generan con `preparar_datos.py`. Este nuevo módulo no modifica esos CSV, no redefine las etiquetas, no abre prueba y no entrena modelos. Al importar un archivo Python puede reaparecer `__pycache__`, que está excluida de Git.

**Siguiente paso, después de revisar esta preparación:** implementar LSTM, CNN 1D y CNN-LSTM con las mismas entradas y dos salidas. Las arquitecturas aprenderán el embedding y usarán una pérdida combinada que se justificará al entrenarlas. 